# Erstellung der Proxy-Komorbiditätscores

## Zweck

Dieses Notebook berechnet für jede ICD-10-Gruppe einen **Proxy-Komorbiditätsscore**
auf Basis der Nebendiagnosen-Verteilung aus dem InEK-Datenbrowser.

Es werden zwei Scores berechnet:

- **Charlsonlike-Score** (Gewichtung nach Quan 2010)
- **Elixhauserlike-Score** (Gewichtung nach Van Walraven)

Da keine Einzelfalldaten vorliegen, wird die Prävalenz der Nebendiagnosen als
Proxy für die Komorbiditätslast verwendet. Eine ausführliche methodische
Erläuterung findet sich in der Arbeit.

## Eingabe

- `icd_systematik_2010.json`: ICD-10-Gruppenhierarchie (erzeugt im ersten Notebook)
- `Datenbrowser Ergebnisse/2024 inkl. Todesfälle/Aufruf nach Nebendiagnosen/{gruppe}.xlsx`:
  Je eine Excel-Datei pro ICD-Gruppe, abgerufen aus dem InEK-Datenbrowser

## Ausgabe

- `comorbidity_lookup.json`: Dictionary mit Charlson- und Elixhauser-Proxyscores
  pro ICD-Gruppe und ICD-Kapitel

## Anmerkung:
Die Datei, die durch dieses Notebook erstellt wird, wurde in dem Repository bereitgestellt, sodass das Notebook nicht ausgeführt werden muss.

In [1]:
import json
from drg_risk import DATA_DIR
import os
import pandas as pd
import numpy as np

# Gewichte und Mapping der Diagnosen und Risikogruppen
from comorbidipy.codemaps.mapping import mapping
from comorbidipy.codemaps.weights import weights

In [2]:
# ICD-Systematik laden — wird zur Iteration über alle Gruppen benötigt
with open(os.path.join(DATA_DIR, "icd_systematik_2010.json")) as f:
    ICD_SYSTEMATIK = json.load(f)

In [3]:
def get_score_category(code: str, score: str):
    """
    Gibt die Risikokategorie eines ICD-Codes für den gewählten Score zurück.

    Verwendet das Quan10-Mapping. Falls kein exakter Treffer gefunden wird,
    wird der Code iterativ von rechts gekürzt (hierarchisches Matching),
    bis ein Treffer vorliegt oder weniger als 3 Zeichen verbleiben.

    Parameters
    ----------
    code : str
        ICD-10-Code (mit oder ohne Punkt, Groß-/Kleinschreibung egal).
    score : str
        "charlson" oder "elixhauser".

    Returns
    -------
    str or None
        Risikokategorie oder None, wenn der Code keiner Kategorie zugeordnet werden kann.
    """

    # Code aufbereiten
    code = code.strip().upper().replace(".", "")

    # Score auswählen
    if score == "charlson":
        mapping_dict_key = "charlson_icd10_quan"
    elif score == "elixhauser":
        mapping_dict_key = "elixhauser_icd10_quan"

    # Code iterativ kürzen, bis Code in Risikokategorie vorhanden oder weniger als 3 Zeichen
    while len(code) >= 3:
        for key, values in mapping[mapping_dict_key].items():
            if code in values:
                # Rückgabe der Risikokategorie
                return key
        code = code[:-1]

    # Code ist in keiner Risikokategorie, keine Rückgabe
    return None

In [4]:
def get_category_max_prevalence(df: dict, score: str) -> dict:
    """
    Berechnet pro Risikokategorie die maximale Nebendiagnosen-Prävalenz einer ICD-Gruppe.

    Die Quelldatei aus dem InEK-Datenbrowser enthält im Sheet "Info" die
    abgefragten ICD-Codes als Filterkriterium. Nur diese Codes werden aus
    dem Sheet "Nebendiagnosen" berücksichtigt.

    Die maximale statt der summierten Prävalenz wird verwendet, um
    Doppelzählungen zu vermeiden: Mehrere Codes derselben Risikokategorie
    können beim selben Fall vorliegen.

    Parameters
    ----------
    df : dict
        Alle Sheets der InEK-Datenbrowser-Excel-Datei (sheet_name=None).
    score : str
        "charlson" oder "elixhauser".

    Returns
    -------
    dict
        Risikokategorie → maximale Prävalenz (in Prozent).
    """

    # Extraktion aus Datei vom InEK-Datenbrowser
    codes = df["Info"]["Diese Datei spiegelt die Daten einer Recherche im InEK Datenbrowser wider, welcher die folgenden Selektionskriterien zugrunde liegen:"].iloc[0].split("(")[1].split(")")[0].split(", ")

    df = df["Nebendiagnosen"][df["Nebendiagnosen"]["Kode"].isin(codes)]

    category_max = {}
    for _, row in df.iterrows():
        # Erhält Risikokategorie
        category = get_score_category(row["Kode"], score=score)
        if category:
            if category in category_max:
                # Wenn die Prävalenz größer ist als vorherige Prävalenzen, Prävalenz als neue größte Prävalenz eintragen
                if row["Prozent (F)"] > category_max[category]:
                    category_max[category] = row["Prozent (F)"]
            else:
                # Prävalenz als ersten Ert eintragen
                category_max[category] = row["Prozent (F)"]
    # Rückgabe der Risikokategorien mit maximaler Prävalenz
    return category_max

In [5]:
def calc_score_lookups(category_max: dict, score: str) -> float:
    """
    Berechnet den Proxy-Komorbiditätsscore aus den maximalen Prävalenzen.

    Score = Σ (Prävalenz_k × Gewicht_k) über alle Risikokategorien k.

    Gewichtungsschema:
    - Charlson: Quan 2010
    - Elixhauser: Van Walraven

    Parameters
    ----------
    category_max : dict
        Risikokategorie → maximale Prävalenz (Ausgabe von get_category_max_prevalence).
    score : str
        "charlson" oder "elixhauser".

    Returns
    -------
    float
        Proxy-Komorbiditätsscore der ICD-Gruppe.
    """
    final_score = 0
    if score == "charlson":
        weight_dict = weights["charlson_icd10_quan"]["quan"]
    elif score == "elixhauser":
        weight_dict = weights["elixhauser_icd10_quan"]["van_walraven"]
    # Über Risikokategorien iterieren
    for key, val in category_max.items():
        # Score = Summe über alle Risikokategorien Gewicht * Prävalenz
        final_score += val * weight_dict.get(key, 0)
    return final_score

In [ ]:
def calc_comorbidity_proxies() -> dict:
    """
    Berechnet die Proxy-Komorbiditätscores für alle ICD-Gruppen.

    Iteriert über die gesamte ICD-10-Systematik und lädt für jede Gruppe
    die zugehörige InEK-Datenbrowser-Datei. Gruppen ohne verfügbare Daten
    (weniger als die Mindestfallzahl im Datenbrowser) erhalten den Score 0.
    Die Kapitel-Ebene wird aus den Gruppen berechnt.

    Returns
    -------
    dict
        ICD-Gruppe → {"charlson": float, "elixhauser": float}
        ICD-Kapitel → {"charlson": float, "elixhauser": float}
    """
    com_lookup = {}
    # Iteriere über alle Kapitel und Gruppen
    for _, chapter in ICD_SYSTEMATIK.items():
        chapter_number = chapter["number"]
        group_fallzahlen = {}
        valid_icd_groups = []
        for _, icd_group in chapter["groups"].items():
            icd_group = icd_group["number"]
            try:
                # Lade Daten vom InEK-Datenbrowser
                df = pd.read_excel(os.path.join(DATA_DIR, "Datenbrowser Ergebnisse", "2024 inkl. Todesfälle", "Aufruf nach Nebendiagnosen", f"{icd_group}.xlsx"), sheet_name = ["Info", "Nebendiagnosen"], engine = "calamine")
                group_fallzahlen[icd_group] = df["Info"]["Unnamed: 1"].iloc[9]
                valid_icd_groups.append(icd_group)
            except FileNotFoundError:
                # Wenn Daten nicht gefunden wurden (wenn zu wenig Fälle vorliegen, wird die Anfrage blockiert), dann wird die Prävalenz auf 0 gesetzt.
                com_lookup[icd_group] = {
                    "charlson": 0,
                    "elixhauser": 0
                }
                group_fallzahlen[icd_group] = 0
                continue
            # maximale Prävalenz pro Risikokategorie
            category_max_elix = get_category_max_prevalence(df, "elixhauser")
            category_max_char = get_category_max_prevalence(df, "charlson")

            # Berechne Proxy-Komorbiditätsscores
            com_lookup[f"Gruppe {icd_group}"] = {
                "charlson": calc_score_lookups(category_max_char, "charlson"),
                "elixhauser": calc_score_lookups(category_max_elix, "elixhauser")
            }

        # Berechnet durchschnittlichen Kapitel Proxy-Komorbiditätsscore basierend auf Gruppen
        weights = [group_fallzahlen[icd_group] for icd_group in valid_icd_groups]
        avg_charlson =      np.average([com_lookup[f"Gruppe {icd_group}"]["charlson"] for icd_group in valid_icd_groups], weights=weights)
        avg_elixhauser =    np.average([com_lookup[f"Gruppe {icd_group}"]["elixhauser"] for icd_group in valid_icd_groups], weights=weights)

        com_lookup[f"Kapitel {chapter_number}"] = {
                "charlson": avg_charlson,
                "elixhauser": avg_elixhauser
            }
    return com_lookup

In [12]:
com_lookup = calc_comorbidity_proxies()

# Verzeichnis als JSON Datei speichern für schnellen Zugriff
with open(os.path.join(DATA_DIR, "comorbidity_lookup.json"), "w") as f:
    json.dump(com_lookup, f, indent=4)